# 🛡️ IoT Device Identification : Hybrid Adversarial Training (CSV + JSON) — **SANS Borderline-SMOTE**

Ce notebook entraîne les modèles sur **DEUX datasets** (CSV + JSON) avec:
- **Dataset CSV** : IPFIX ML Instances (12 foyers, 18 classes IoT)
- **Dataset JSON** : IPFIX Records (3 mois, 17 classes IoT)
- **Curriculum Learning** (2 phases: Standard → Adversarial)
- **Early Stopping par phase**
- **Crash Test** après chaque phase (bénignes + adversaires + mélangé)
- **Macro F1-Score** et métriques par classe pour chaque phase
- **Plots de présentation** générés automatiquement à chaque phase
- **⚠️ SANS Borderline-SMOTE** — pour comparaison avec la version avec équilibrage

Tous les résultats et plots sont sauvegardés sur **Google Drive**.

# 🛠️ Phase 1 : Configuration de l'Environnement (Setup)

In [1]:
# ─── Cell 1: Setup ───────────────────────────────────────────────────────────
# Mount Google Drive FIRST so the paths in Cell 2 resolve correctly
from google.colab import drive
drive.mount('/content/drive')

# Clone the repository (pull latest if already cloned)
import os
if os.path.exists('/content/pfe'):
    !cd /content/pfe && git pull
else:
    !git clone https://github.com/yacinemkk/pfe.git /content/pfe

%cd /content/pfe

# Install dependencies
!pip install -q torch torchvision tqdm numpy pandas scikit-learn matplotlib xgboost psutil

Mounted at /content/drive
Cloning into '/content/pfe'...
remote: Enumerating objects: 597, done.
remote: Counting objects: 100% (320/320), done.
remote: Compressing objects: 100% (232/232), done.
remote: Total 597 (delta 140), reused 246 (delta 83), pack-reused 277 (from 1)
Receiving objects: 100% (597/597), 14.19 MiB | 18.44 MiB/s, done.
Resolving deltas: 100% (260/260), done.
/content/pfe


# ⚙️ Phase 2 : Configuration & Pipeline de Données

In [2]:
# ─── Cell 2: Configure ───────────────────────────────────────────────────────
# ⚠️  UPDATE THESE PATHS before pressing Run All.

# Path to your JSON IPFIX files on Google Drive.
# Expected structure:
#   JSON_DATA_DIR/
#     20-01-31(1)/ipfix_202001_fixed.json
#     20-03-31/ipfix_202003.json
#     20-04-30/ipfix_202004.json
JSON_DATA_DIR = '/content/drive/MyDrive/PFE/IPFIX_Records'

# Path to your CSV IPFIX ML Instances on Google Drive.
# Expected structure:
#   CSV_DATA_DIR/
#     home1_labeled.csv
#     home2_labeled.csv
#     ...
CSV_DATA_DIR = '/content/drive/MyDrive/PFE/IPFIX_ML_Instances'

# Where to save trained models, results.json, history.json, and plots.
# This folder is on Drive — it persists after the runtime disconnects.
DRIVE_RESULTS_DIR = '/content/drive/MyDrive/PFE/results'

# Datasets to train on: 'csv', 'json', or 'both'
DATASETS = 'both'  # Options: 'csv', 'json', 'both'

# ─── Training hyperparameters ─────────────────────────────────────────────────
SEQ_LENGTH   = 10      # Sequence length (timesteps per sample)
EPOCHS       = 30      # Total epochs (split across 2 phases)
BATCH_SIZE   = 64      # Batch size
ADV_METHOD   = 'hybrid' # Adversarial method: none, feature, pgd, fgsm, hybrid
ADV_RATIO    = 0.2     # Ratio of adversarial samples during training
MAX_FILES    = None    # Max CSV files to load (None = all). Use small int for testing.
MAX_RECORDS  = None    # Max JSON records to load (None = all). Use small int for testing.

# ─── RAM Optimization ─────────────────────────────────────────────────────────
STRIDE = 10             # Larger stride = fewer sequences = less RAM
EVAL_SUBSAMPLE = 1000  # Max samples for adversarial eval
EVAL_BATCH_SIZE = 32   # Smaller batch for evaluation (saves RAM)

# Create results directory if needed
os.makedirs(DRIVE_RESULTS_DIR, exist_ok=True)

print(f'CSV data:     {CSV_DATA_DIR}')
print(f'JSON data:    {JSON_DATA_DIR}')
print(f'Results dir:  {DRIVE_RESULTS_DIR}')
print(f'Datasets:     {DATASETS}')
print(f'Seq length:   {SEQ_LENGTH}')
print(f'Epochs:       {EPOCHS} (Phase 1 + Phase 2)')
print(f'Batch size:   {BATCH_SIZE}')
print(f'Adv method:   {ADV_METHOD}')
print(f'Adv ratio:    {ADV_RATIO}')

# Verify CSV data exists
import glob
csv_files = glob.glob(f'{CSV_DATA_DIR}/home*_labeled.csv')
print(f'\nFound {len(csv_files)} CSV file(s):')
for f in sorted(csv_files)[:5]:
    size_mb = os.path.getsize(f) / (1024**2)
    print(f'  {os.path.basename(f)} ({size_mb:.1f} MB)')
if len(csv_files) > 5:
    print(f'  ... and {len(csv_files) - 5} more')

# Verify JSON data exists
json_files = glob.glob(f'{JSON_DATA_DIR}/**/*.json', recursive=True)
print(f'\nFound {len(json_files)} JSON file(s):')
for f in json_files:
    size_gb = os.path.getsize(f) / (1024**3)
    print(f'  {os.path.basename(f)} ({size_gb:.1f} GB)')

CSV data:     /content/drive/MyDrive/PFE/IPFIX_ML_Instances
JSON data:    /content/drive/MyDrive/PFE/IPFIX_Records
Results dir:  /content/drive/MyDrive/PFE/results
Datasets:     both
Seq length:   10
Epochs:       30 (Phase 1 + Phase 2)
Batch size:   64
Adv method:   hybrid
Adv ratio:    0.2

Found 12 CSV file(s):
  home10_labeled.csv (1099.6 MB)
  home11_labeled.csv (215.8 MB)
  home12_labeled.csv (332.2 MB)
  home1_labeled.csv (245.0 MB)
  home2_labeled.csv (297.0 MB)
  ... and 7 more

Found 3 JSON file(s):
  ipfix_202003.json (5.7 GB)
  ipfix_202004.json (5.3 GB)
  ipfix_202001_fixed.json (6.6 GB)


In [3]:
# ─── RAM Monitoring & Cleanup Utilities ───────────────────────────────────────
import gc
import psutil
import torch
import os

def get_memory_usage():
    process = psutil.Process(os.getpid())
    ram_gb = process.memory_info().rss / (1024**3)
    gpu_gb = 0
    if torch.cuda.is_available():
        gpu_gb = torch.cuda.memory_allocated() / (1024**3)
    return ram_gb, gpu_gb

def log_memory(label=''):
    ram_gb, gpu_gb = get_memory_usage()
    print(f'  [RAM {label}] {ram_gb:.2f} GB | [GPU {label}] {gpu_gb:.2f} GB')

def aggressive_cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
    gc.collect()
    ram_gb, gpu_gb = get_memory_usage()
    print(f'  [Cleanup] RAM: {ram_gb:.2f} GB | GPU: {gpu_gb:.2f} GB')

print('RAM monitoring utilities loaded.')
log_memory('startup')

RAM monitoring utilities loaded.
  [RAM startup] 0.56 GB | [GPU startup] 0.00 GB


# 🧠 Phase 3a : Entraînement sur CSV — Un Modèle par Cellule

**Workflow optimisé pour la RAM:**
- Les datasets CSV et JSON ne sont **PAS chargés en même temps** (manque de RAM)
- Phase 3a: Charger CSV → Entraîner chaque modèle dans une cellule indépendante
- Nettoyage RAM unique entre les deux phases
- Phase 3b: Charger JSON (labelisation via adresses MAC) → Entraîner chaque modèle

**Modèles:** LSTM → BiLSTM → CNN-LSTM → XGBoost-LSTM → Transformer → CNN-BiLSTM-Transformer

Pour changer de dataset, modifiez `DATASETS` dans la Cell 2.

## 📦 Chargement du Dataset CSV
Affiche les informations détaillées du dataset CSV avant l'entraînement.


In [4]:
def load_and_display_csv_dataset(csv_data_dir, seq_length=10, stride=10, save_dir=None):
    """Charge COMPLÈTEMENT le dataset CSV, affiche les infos, et sauvegarde sur Drive."""
    import sys
    import gc
    import numpy as np
    import pickle

    sys.path.insert(0, '/content/pfe')

    print("\n" + "=" * 70)
    print("  CHARGEMENT COMPLET DU DATASET CSV")
    print("=" * 70)

    print(f"\n  Répertoire : {csv_data_dir}")
    print(f"  Seq length : {seq_length} | Stride : {stride}")

    csv_files = sorted(glob.glob(f'{csv_data_dir}/home*_labeled.csv'))
    print(f"\n  Fichiers CSV trouvés : {len(csv_files)}")

    for f in csv_files:
        size_mb = os.path.getsize(f) / (1024**2)
        print(f"    {os.path.basename(f):<30s} : {size_mb:>10.1f} MB")

    total_gb = sum(os.path.getsize(f) for f in csv_files) / (1024**3)
    print(f"\n  Taille totale : {total_gb:.2f} GB")

    # Charger le dataset complet via le vrai pipeline CSV
    print("\n  Chargement via le pipeline CSV (IoTDataProcessor)...")
    from src.data.preprocessor import IoTDataProcessor

    processor = IoTDataProcessor()
    result = processor.process_all(
        max_files=None,
        data_dir=csv_data_dir,
        seq_length=seq_length,
        stride=stride,
        apply_balancing=False,
    )

    X_train, X_val, X_test, y_train, y_val, y_test, features, scaler, label_encoder = result
    n_continuous = len(features)

    print(f"\n  {'='*70}")
    print(f"  RÉSULTAT DU CHARGEMENT")
    print(f"  {'='*70}")
    print(f"    Features ({n_continuous}) : {features[:5]}...")
    print(f"    Classes ({len(label_encoder.classes_)}) : {list(label_encoder.classes_)}")
    print(f"\n  Shapes des séquences (seq_length={seq_length}, stride={stride}) :")
    print(f"    Train : {X_train.shape}  →  {len(X_train):,} séquences")
    print(f"    Val   : {X_val.shape}  →  {len(X_val):,} séquences")
    print(f"    Test  : {X_test.shape}  →  {len(X_test):,} séquences")
    print(f"    Total : {len(X_train) + len(X_val) + len(X_test):,} séquences")

    print(f"\n  Distribution des classes (train) :")
    for cls in label_encoder.classes_:
        cls_id = label_encoder.transform([cls])[0]
        count = int(np.sum(y_train == cls_id))
        bar = '█' * max(1, count // 50)
        print(f"    {cls:<30s} : {count:>6,}  {bar}")

    # ─── Sauvegarder sur Drive ────────────────────────────────────────────
    if save_dir:
        os.makedirs(save_dir, exist_ok=True)
        print(f"\n  💾 Sauvegarde du dataset CSV pré-traité sur Drive...")
        print(f"     Répertoire : {save_dir}")

        np.save(f'{save_dir}/X_train.npy', X_train)
        np.save(f'{save_dir}/X_val.npy', X_val)
        np.save(f'{save_dir}/X_test.npy', X_test)
        np.save(f'{save_dir}/y_train.npy', y_train)
        np.save(f'{save_dir}/y_val.npy', y_val)
        np.save(f'{save_dir}/y_test.npy', y_test)

        with open(f'{save_dir}/csv_metadata.pkl', 'wb') as f:
            pickle.dump({
                'features': features,
                'scaler': scaler,
                'label_encoder': label_encoder,
                'n_continuous': n_continuous,
                'seq_length': seq_length,
                'stride': stride,
            }, f)

        # Marker file to indicate preprocessing is complete
        with open(f'{save_dir}/csv_ready', 'w') as f:
            f.write('ready')

        saved_gb = (X_train.nbytes + X_val.nbytes + X_test.nbytes +
                    y_train.nbytes + y_val.nbytes + y_test.nbytes) / (1024**3)
        print(f"  ✅ Dataset CSV sauvegardé ({saved_gb:.2f} GB)")
        print(f"     Fichiers : X_train, X_val, X_test, y_train, y_val, y_test, csv_metadata.pkl")

    print(f"\n  {'='*70}")
    print(f"  ✅ Dataset CSV chargé complètement en RAM")
    print(f"  {'='*70}\n")

    return {
        'X_train': X_train, 'X_val': X_val, 'X_test': X_test,
        'y_train': y_train, 'y_val': y_val, 'y_test': y_test,
        'features': features, 'scaler': scaler,
        'label_encoder': label_encoder, 'n_continuous': n_continuous
    }

# ─── CSV preprocessing directory on Drive ─────────────────────────────
CSV_PREPROCESSED_DIR = f'{DRIVE_RESULTS_DIR}/preprocessed/csv'

if DATASETS in ['csv', 'both']:
    csv_data = load_and_display_csv_dataset(
        CSV_DATA_DIR, seq_length=SEQ_LENGTH, stride=STRIDE,
        save_dir=CSV_PREPROCESSED_DIR
    )
else:
    csv_data = None
    print('Skipping CSV dataset loading — DATASETS is not csv or both')



  CHARGEMENT COMPLET DU DATASET CSV

  Répertoire : /content/drive/MyDrive/PFE/IPFIX_ML_Instances
  Seq length : 10 | Stride : 10

  Fichiers CSV trouvés : 12
    home10_labeled.csv             :     1099.6 MB
    home11_labeled.csv             :      215.8 MB
    home12_labeled.csv             :      332.2 MB
    home1_labeled.csv              :      245.0 MB
    home2_labeled.csv              :      297.0 MB
    home3_labeled.csv              :      354.7 MB
    home4_labeled.csv              :     1541.7 MB
    home5_labeled.csv              :      366.6 MB
    home6_labeled.csv              :      932.0 MB
    home7_labeled.csv              :      527.0 MB
    home8_labeled.csv              :      317.2 MB
    home9_labeled.csv              :      238.2 MB

  Taille totale : 6.32 GB

  Chargement via le pipeline CSV (IoTDataProcessor)...
CSV-NATIVE IPFIX ML INSTANCES PREPROCESSOR (Pipeline 4 Etapes)

[ETAPE 1] Filtrage et adaptation au SDN...
  1.1 Chargement des fichiers CSV...
 

In [5]:
# ─── Training Helper Function ────────────────────────────────────────────────
import subprocess
import sys

def train_model(MODEL, dataset_type, data_dir):
    """Train a single model on a single dataset (WITHOUT Borderline-SMOTE)."""
    print(f'\n{"="*60}')
    print(f'  Training: {MODEL.upper()} on {dataset_type.upper()} (NO SMOTE)')
    print(f'{"="*60}\n')

    # ─── GPU Check ───────────────────────────────────────────────────────
    try:
        import torch
        if torch.cuda.is_available():
            gpu_count = torch.cuda.device_count()
            gpu_name = torch.cuda.get_device_name(0)
            gpu_mem = torch.cuda.get_device_properties(0).total_mem / (1024**3)
            print(f'  🚀 CUDA Available: YES')
            print(f'  📌 GPU: {gpu_name} ({gpu_mem:.1f} GB)')
            print(f'  🔢 GPU Count: {gpu_count}')
            print(f'  ⚡ CUDA Version: {torch.version.cuda}')
        else:
            print(f'  ⚠️  CUDA NOT available — training on CPU')
            print(f'  💡 Tip: In Colab, go to Runtime > Change runtime type > Hardware accelerator > GPU')
    except ImportError:
        print(f'  ⚠️  PyTorch not installed')
    print()

    model_dir = f'{DRIVE_RESULTS_DIR}/models/{MODEL}_{dataset_type}_nor'
    max_arg = f'--max_files {MAX_FILES}' if dataset_type == 'csv' and MAX_FILES else ''
    max_arg = f'--max_records {MAX_RECORDS}' if dataset_type == 'json' and MAX_RECORDS else max_arg

    # ─── Preprocessed data directory ─────────────────────────────────────
    preprocessed_dir = f'{DRIVE_RESULTS_DIR}/preprocessed/{dataset_type}'
    preprocessed_arg = f'--preprocessed_dir "{preprocessed_dir}"'

    cmd = (
        f'python train_adversarial.py '
        f'--model {MODEL} '
        f'--seq_length {SEQ_LENGTH} '
        f'--adv_method {ADV_METHOD} '
        f'--adv_ratio {ADV_RATIO} '
        f'--epochs {EPOCHS} '
        f'--batch_size {BATCH_SIZE} '
        f'--dataset {dataset_type} '
        f'--data_dir "{data_dir}" '
        f'--results_dir "{model_dir}" '
        f'--phase_checkpoints '
        f'--no_balancing '
        f'--use_class_weights '
        f'{preprocessed_arg} '
        f'{max_arg}'
    )
    print(f'  CMD: {cmd[:80]}...')
    print(f'\n  ── Training Output (live) ──────────────────────────────\n')

    # Stream output in real-time so epochs are visible as they happen
    process = subprocess.Popen(
        cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE,
        text=True, bufsize=1, universal_newlines=True
    )

    # Read stdout line by line in real-time
    for line in process.stdout:
        print(line, end='')
        sys.stdout.flush()

    # Wait for process to finish and capture stderr
    _, stderr = process.communicate()
    returncode = process.returncode

    print(f'\n  ─────────────────────────────────────────────────────────\n')

    if returncode == 0:
        print(f'  ✅ {MODEL.upper()} SAVED to {model_dir}')
    else:
        print(f'  ❌ {MODEL.upper()} FAILED (exit code: {returncode})')
        if stderr:
            print(f'\n  STDERR:\n{stderr[-3000:]}')

def verify_drive_save(MODEL, dataset_type):
    """Verify files are saved to Drive."""
    _dir = f'{DRIVE_RESULTS_DIR}/models/{MODEL}_{dataset_type}_nor'
    _files = glob.glob(f'{_dir}/*')
    if _files:
        print(f'  ✅ Drive/{MODEL}_{dataset_type}_nor: {len(_files)} file(s) saved')
        for _f in sorted(_files)[:5]:
            print(f'      - {os.path.basename(_f)}')
    else:
        print(f'  ⚠️  Drive/{MODEL}_{dataset_type}_nor: no files found')


In [6]:
# ─── MODEL 1: LSTM on CSV ─────────────────────────────────────────────
MODEL = 'lstm'
print(f'\n{"#"*80}')
print(f'  PHASE 3a — {MODEL.upper()} on CSV')
print(f'{"#"*80}\n')

log_memory(f'before_{MODEL}_csv')

if DATASETS in ['csv', 'both']:
    train_model(MODEL, 'csv', CSV_DATA_DIR)
    verify_drive_save(MODEL, 'csv')
else:
    print('Skipping CSV dataset')

print(f'\n✅ {MODEL.upper()} on CSV DONE')



################################################################################
  PHASE 3a — LSTM on CSV
################################################################################

  [RAM before_lstm_csv] 3.48 GB | [GPU before_lstm_csv] 0.00 GB

  Training: LSTM on CSV (NO SMOTE)



AttributeError: 'torch._C._CudaDeviceProperties' object has no attribute 'total_mem'

In [ ]:
# ─── MODEL 2: BiLSTM on CSV ───────────────────────────────────────────
MODEL = 'bilstm'
print(f'\n{"#"*80}')
print(f'  PHASE 3a — {MODEL.upper()} on CSV')
print(f'{"#"*80}\n')

log_memory(f'before_{MODEL}_csv')

if DATASETS in ['csv', 'both']:
    train_model(MODEL, 'csv', CSV_DATA_DIR)
    verify_drive_save(MODEL, 'csv')
else:
    print('Skipping CSV dataset')

print(f'\n✅ {MODEL.upper()} on CSV DONE')


In [ ]:
# ─── MODEL 3: CNN-LSTM on CSV ───────────────────────────────────
MODEL = 'cnn_lstm'
print(f'\n{"#"*80}')
print(f'  PHASE 3a — {MODEL.upper()} on CSV')
print(f'{"#"*80}\n')

log_memory(f'before_{MODEL}_csv')

if DATASETS in ['csv', 'both']:
    train_model(MODEL, 'csv', CSV_DATA_DIR)
    verify_drive_save(MODEL, 'csv')
else:
    print('Skipping CSV dataset')

print(f'\n✅ {MODEL.upper()} on CSV DONE')


In [ ]:
# ─── MODEL 4: XGBoost-LSTM on CSV ───────────────────────────────
MODEL = 'xgboost_lstm'
print(f'\n{"#"*80}')
print(f'  PHASE 3a — {MODEL.upper()} on CSV')
print(f'{"#"*80}\n')

log_memory(f'before_{MODEL}_csv')

if DATASETS in ['csv', 'both']:
    train_model(MODEL, 'csv', CSV_DATA_DIR)
    verify_drive_save(MODEL, 'csv')
else:
    print('Skipping CSV dataset')

print(f'\n✅ {MODEL.upper()} on CSV DONE')


## 🔍 Vérification de la Tokenization (CSV)
Étape importante avant d'entraîner les modèles **Transformer** et **CNN-BiLSTM-Transformer**.


In [ ]:
def verify_tokenization_for_transformers(seq_length=10, input_size=37):
    """Vérifie que la tokenization fonctionne avec les modèles transformer."""
    import torch
    import numpy as np

    sys.path.insert(0, '/content/pfe')

    from src.data.tokenizer import IoTTokenizer, SimpleTokenizer, create_tokenizer
    from src.models.transformer import TransformerClassifier, NLPTransformerClassifier
    from src.models.cnn_bilstm_transformer import CNNBiLSTMTransformerClassifier
    from config.config import CNN_BILSTM_TRANSFORMER_CONFIG, TRANSFORMER_CONFIG

    print("\n" + "=" * 70)
    print("  VÉRIFICATION DE LA TOKENIZATION")
    print("=" * 70)

    # 1. Créer le tokenizer
    print("\n  [1] Création du tokenizer...")
    tokenizer = create_tokenizer()
    print(f"      Tokenizer : {type(tokenizer).__name__}")

    # 2. Simuler des données d'entrée pour les tests
    print("\n  [2] Simulation des données d'entrée...")
    num_classes = 18  # CSV classes
    batch_size = 2

    # Données brutes (batch, seq_len, input_size)
    X_raw = np.random.randn(batch_size, seq_length, input_size).astype(np.float32)

    # 3. Tester TransformerClassifier (input_size-based, pas de tokenization)
    print("\n  [3] Vérification avec TransformerClassifier...")
    transformer_model = TransformerClassifier(
        input_size=input_size,
        num_classes=num_classes,
        seq_length=seq_length,
        config=TRANSFORMER_CONFIG
    )
    x_tensor = torch.FloatTensor(X_raw)
    try:
        out = transformer_model(x_tensor)
        print(f"      ✅ TransformerClassifier : input {x_tensor.shape} → output {out.shape}")
        transformer_ok = True
    except Exception as e:
        print(f"      ❌ TransformerClassifier échec : {e}")
        transformer_ok = False

    # 4. Tester NLPTransformerClassifier (token-based)
    print("\n  [4] Vérification avec NLPTransformerClassifier...")
    vocab_size = 52000
    nlp_transformer = NLPTransformerClassifier(
        vocab_size=vocab_size,
        num_classes=num_classes,
        max_seq_length=seq_length
    )
    x_tok_tensor = torch.randint(0, vocab_size, (batch_size, seq_length))
    try:
        out = nlp_transformer(x_tok_tensor)
        print(f"      ✅ NLPTransformerClassifier : input {x_tok_tensor.shape} → output {out.shape}")
        nlp_ok = True
    except Exception as e:
        print(f"      ❌ NLPTransformerClassifier échec : {e}")
        nlp_ok = False

    # 5. Tester CNNBiLSTMTransformerClassifier
    print("\n  [5] Vérification avec CNNBiLSTMTransformerClassifier...")
    hybrid_model = CNNBiLSTMTransformerClassifier(
        input_size=input_size,
        num_classes=num_classes,
        seq_length=seq_length,
        config=CNN_BILSTM_TRANSFORMER_CONFIG
    )
    try:
        out = hybrid_model(x_tensor)
        print(f"      ✅ CNNBiLSTMTransformerClassifier : input {x_tensor.shape} → output {out.shape}")
        hybrid_ok = True
    except Exception as e:
        print(f"      ❌ CNNBiLSTMTransformerClassifier échec : {e}")
        hybrid_ok = False

    # Résumé
    print("\n" + "=" * 70)
    print("  RÉSUMÉ DE LA VÉRIFICATION")
    print("=" * 70)
    print(f"    Tokenizer créé           : {'✅ OK' if tokenizer else '❌ Échec'}")
    print(f"    Vocabulaire size         : {vocab_size:,} tokens")
    print(f"    TransformerClassifier    : {'✅ Compatible (features brutes)' if transformer_ok else '❌ Incompatible'}")
    print(f"    NLPTransformerClassifier : {'✅ Compatible (tokens)' if nlp_ok else '❌ Incompatible'}")
    print(f"    CNNBiLSTMTransformer     : {'✅ Compatible (features brutes)' if hybrid_ok else '❌ Incompatible'}")
    print("=" * 70 + "\n")

    return {
        'tokenizer': tokenizer,
        'transformer_ok': transformer_ok,
        'nlp_transformer_ok': nlp_ok,
        'hybrid_ok': hybrid_ok,
        'vocab_size': vocab_size
    }

if DATASETS in ['csv', 'both']:
    tokenization_results_csv = verify_tokenization_for_transformers(seq_length=SEQ_LENGTH, input_size=37)
else:
    print('Skipping tokenization verification for CSV')


In [ ]:
# ─── MODEL 5: Transformer on CSV ────────────────────────────────
MODEL = 'transformer'
print(f'\n{"#"*80}')
print(f'  PHASE 3a — {MODEL.upper()} on CSV')
print(f'{"#"*80}\n')

log_memory(f'before_{MODEL}_csv')

if DATASETS in ['csv', 'both']:
    train_model(MODEL, 'csv', CSV_DATA_DIR)
    verify_drive_save(MODEL, 'csv')
else:
    print('Skipping CSV dataset')

print(f'\n✅ {MODEL.upper()} on CSV DONE')


In [ ]:
# ─── MODEL 6: CNN-BiLSTM-Transformer on CSV ─────────────────────
MODEL = 'cnn_bilstm_transformer'
print(f'\n{"#"*80}')
print(f'  PHASE 3a — {MODEL.upper()} on CSV')
print(f'{"#"*80}\n')

log_memory(f'before_{MODEL}_csv')

if DATASETS in ['csv', 'both']:
    train_model(MODEL, 'csv', CSV_DATA_DIR)
    verify_drive_save(MODEL, 'csv')
else:
    print('Skipping CSV dataset')

print(f'\n✅ {MODEL.upper()} on CSV DONE')

print(f'\n{"="*80}')
print('  ✅ PHASE 3a COMPLETE — ALL 6 MODELS TRAINED ON CSV')
print(f'{"="*80}')


# 🧹 Nettoyage RAM entre CSV et JSON

Exécutez cette cellule une seule fois après avoir terminé tous les modèles CSV.

In [ ]:
# ─── CLEANUP RAM BEFORE JSON PHASE ──────────────────────────────────────
print('\n' + '='*80)
print('  🧹 CLEANING RAM BEFORE JSON PHASE...')
print('='*80 + '\n')

aggressive_cleanup()

print('\n✅ RAM cleaned. Ready for JSON phase.')


# 🧠 Phase 3b : Entraînement sur JSON — Un Modèle par Cellule

**Le JSON est chargé avec labelisation par adresses MAC** (sourceMacAddress / destinationMacAddress).
Chaque cellule entraîne un modèle sur JSON indépendamment.

## 📦 Chargement du Dataset JSON
Affiche les informations détaillées du dataset JSON chargé avant l'entraînement.


In [ ]:
def load_and_display_json_dataset(json_data_dir, seq_length=10, stride=10, max_records=None, save_dir=None):
    """Charge COMPLÈTEMENT le dataset JSON, affiche les infos, et sauvegarde sur Drive."""
    import sys
    import gc
    import numpy as np
    import pickle

    sys.path.insert(0, '/content/pfe')

    print("\n" + "=" * 70)
    print("  CHARGEMENT COMPLET DU DATASET JSON")
    print("=" * 70)

    print(f"\n  Répertoire : {json_data_dir}")
    print(f"  Seq length : {seq_length} | Stride : {stride}")
    print(f"  Max records: {max_records if max_records else 'Tous'}")

    json_files = sorted(glob.glob(f'{json_data_dir}/**/*.json', recursive=True))
    print(f"\n  Fichiers JSON trouvés : {len(json_files)}")

    for f in json_files:
        size_gb = os.path.getsize(f) / (1024**3)
        print(f"    {os.path.basename(f):<30s} : {size_gb:>10.2f} GB")

    total_gb = sum(os.path.getsize(f) for f in json_files) / (1024**3)
    print(f"\n  Taille totale : {total_gb:.2f} GB")

    # Charger le dataset complet via le vrai pipeline JSON
    print("\n  Chargement via le pipeline JSON (JsonIoTDataProcessor)...")
    from src.data.json_preprocessor import JsonIoTDataProcessor

    processor = JsonIoTDataProcessor()
    result = processor.process_all(
        data_dir=json_data_dir,
        seq_length=seq_length,
        stride=stride,
        max_records=max_records,
        apply_balancing=False,
    )

    X_train, X_val, X_test, y_train, y_val, y_test, features, scaler, label_encoder = result
    n_continuous = 36  # JSON pipeline features

    print(f"\n  {'='*70}")
    print(f"  RÉSULTAT DU CHARGEMENT")
    print(f"  {'='*70}")
    print(f"    Features ({len(features)}) : {features[:5]}...")
    print(f"    Classes ({len(label_encoder.classes_)}) : {list(label_encoder.classes_)}")
    print(f"\n  Shapes des séquences (seq_length={seq_length}, stride={stride}) :")
    print(f"    Train : {X_train.shape}  →  {len(X_train):,} séquences")
    print(f"    Val   : {X_val.shape}  →  {len(X_val):,} séquences")
    print(f"    Test  : {X_test.shape}  →  {len(X_test):,} séquences")
    print(f"    Total : {len(X_train) + len(X_val) + len(X_test):,} séquences")

    print(f"\n  Distribution des classes (train) :")
    for cls in label_encoder.classes_:
        cls_id = label_encoder.transform([cls])[0]
        count = int(np.sum(y_train == cls_id))
        bar = '█' * max(1, count // 50)
        print(f"    {cls:<30s} : {count:>6,}  {bar}")

    # ─── Sauvegarder sur Drive ────────────────────────────────────────────
    if save_dir:
        os.makedirs(save_dir, exist_ok=True)
        print(f"\n  💾 Sauvegarde du dataset JSON pré-traité sur Drive...")
        print(f"     Répertoire : {save_dir}")

        np.save(f'{save_dir}/X_train.npy', X_train)
        np.save(f'{save_dir}/X_val.npy', X_val)
        np.save(f'{save_dir}/X_test.npy', X_test)
        np.save(f'{save_dir}/y_train.npy', y_train)
        np.save(f'{save_dir}/y_val.npy', y_val)
        np.save(f'{save_dir}/y_test.npy', y_test)

        with open(f'{save_dir}/json_metadata.pkl', 'wb') as f:
            pickle.dump({
                'features': features,
                'scaler': scaler,
                'label_encoder': label_encoder,
                'n_continuous': n_continuous,
                'seq_length': seq_length,
                'stride': stride,
            }, f)

        # Marker file to indicate preprocessing is complete
        with open(f'{save_dir}/json_ready', 'w') as f:
            f.write('ready')

        saved_gb = (X_train.nbytes + X_val.nbytes + X_test.nbytes +
                    y_train.nbytes + y_val.nbytes + y_test.nbytes) / (1024**3)
        print(f"  ✅ Dataset JSON sauvegardé ({saved_gb:.2f} GB)")
        print(f"     Fichiers : X_train, X_val, X_test, y_train, y_val, y_test, json_metadata.pkl")

    print(f"\n  {'='*70}")
    print(f"  ✅ Dataset JSON chargé complètement en RAM")
    print(f"  {'='*70}\n")

    return {
        'X_train': X_train, 'X_val': X_val, 'X_test': X_test,
        'y_train': y_train, 'y_val': y_val, 'y_test': y_test,
        'features': features, 'scaler': scaler,
        'label_encoder': label_encoder, 'n_continuous': n_continuous
    }

# ─── JSON preprocessing directory on Drive ─────────────────────────────
JSON_PREPROCESSED_DIR = f'{DRIVE_RESULTS_DIR}/preprocessed/json'

if DATASETS in ['json', 'both']:
    json_data = load_and_display_json_dataset(
        JSON_DATA_DIR, seq_length=SEQ_LENGTH, stride=STRIDE,
        max_records=MAX_RECORDS, save_dir=JSON_PREPROCESSED_DIR
    )
else:
    json_data = None
    print('Skipping JSON dataset loading — DATASETS is not json or both')


In [ ]:
# ─── MODEL 1: LSTM on JSON ─────────────────────────────────────────────
MODEL = 'lstm'
print(f'\n{"#"*80}')
print(f'  PHASE 3b — {MODEL.upper()} on JSON (MAC address labeling)')
print(f'{"#"*80}\n')

log_memory(f'before_{MODEL}_json')

if DATASETS in ['json', 'both']:
    train_model(MODEL, 'json', JSON_DATA_DIR)
    verify_drive_save(MODEL, 'json')
else:
    print('Skipping JSON dataset')

print(f'\n✅ {MODEL.upper()} on JSON DONE')


In [ ]:
# ─── MODEL 2: BiLSTM on JSON ───────────────────────────────────────────
MODEL = 'bilstm'
print(f'\n{"#"*80}')
print(f'  PHASE 3b — {MODEL.upper()} on JSON (MAC address labeling)')
print(f'{"#"*80}\n')

log_memory(f'before_{MODEL}_json')

if DATASETS in ['json', 'both']:
    train_model(MODEL, 'json', JSON_DATA_DIR)
    verify_drive_save(MODEL, 'json')
else:
    print('Skipping JSON dataset')

print(f'\n✅ {MODEL.upper()} on JSON DONE')


In [ ]:
# ─── MODEL 3: CNN-LSTM on JSON ───────────────────────────────────
MODEL = 'cnn_lstm'
print(f'\n{"#"*80}')
print(f'  PHASE 3b — {MODEL.upper()} on JSON (MAC address labeling)')
print(f'{"#"*80}\n')

log_memory(f'before_{MODEL}_json')

if DATASETS in ['json', 'both']:
    train_model(MODEL, 'json', JSON_DATA_DIR)
    verify_drive_save(MODEL, 'json')
else:
    print('Skipping JSON dataset')

print(f'\n✅ {MODEL.upper()} on JSON DONE')


In [ ]:
# ─── MODEL 4: XGBoost-LSTM on JSON ───────────────────────────────
MODEL = 'xgboost_lstm'
print(f'\n{"#"*80}')
print(f'  PHASE 3b — {MODEL.upper()} on JSON (MAC address labeling)')
print(f'{"#"*80}\n')

log_memory(f'before_{MODEL}_json')

if DATASETS in ['json', 'both']:
    train_model(MODEL, 'json', JSON_DATA_DIR)
    verify_drive_save(MODEL, 'json')
else:
    print('Skipping JSON dataset')

print(f'\n✅ {MODEL.upper()} on JSON DONE')


## 🔍 Vérification de la Tokenization (JSON)
Étape importante avant d'entraîner les modèles **Transformer** et **CNN-BiLSTM-Transformer**.


In [ ]:
def verify_tokenization_for_transformers_json(seq_length=10, input_size=36):
    """Vérifie que la tokenization fonctionne avec les modèles transformer (JSON pipeline)."""
    import torch
    import numpy as np

    sys.path.insert(0, '/content/pfe')

    from src.data.tokenizer import IoTTokenizer, SimpleTokenizer, create_tokenizer
    from src.models.transformer import TransformerClassifier, NLPTransformerClassifier
    from src.models.cnn_bilstm_transformer import CNNBiLSTMTransformerClassifier
    from config.config import CNN_BILSTM_TRANSFORMER_CONFIG, TRANSFORMER_CONFIG

    print("\n" + "=" * 70)
    print("  VÉRIFICATION DE LA TOKENIZATION (JSON)")
    print("=" * 70)

    # 1. Créer le tokenizer
    print("\n  [1] Création du tokenizer...")
    tokenizer = create_tokenizer()
    print(f"      Tokenizer : {type(tokenizer).__name__}")

    # 2. Simuler des données d'entrée pour les tests
    print("\n  [2] Simulation des données d'entrée...")
    num_classes = 17  # JSON classes
    batch_size = 2

    # Données brutes (batch, seq_len, input_size)
    X_raw = np.random.randn(batch_size, seq_length, input_size).astype(np.float32)

    # 3. Tester TransformerClassifier (input_size-based, pas de tokenization)
    print("\n  [3] Vérification avec TransformerClassifier...")
    transformer_model = TransformerClassifier(
        input_size=input_size,
        num_classes=num_classes,
        seq_length=seq_length,
        config=TRANSFORMER_CONFIG
    )
    x_tensor = torch.FloatTensor(X_raw)
    try:
        out = transformer_model(x_tensor)
        print(f"      ✅ TransformerClassifier : input {x_tensor.shape} → output {out.shape}")
        transformer_ok = True
    except Exception as e:
        print(f"      ❌ TransformerClassifier échec : {e}")
        transformer_ok = False

    # 4. Tester NLPTransformerClassifier (token-based)
    print("\n  [4] Vérification avec NLPTransformerClassifier...")
    vocab_size = 52000
    nlp_transformer = NLPTransformerClassifier(
        vocab_size=vocab_size,
        num_classes=num_classes,
        max_seq_length=seq_length
    )
    x_tok_tensor = torch.randint(0, vocab_size, (batch_size, seq_length))
    try:
        out = nlp_transformer(x_tok_tensor)
        print(f"      ✅ NLPTransformerClassifier : input {x_tok_tensor.shape} → output {out.shape}")
        nlp_ok = True
    except Exception as e:
        print(f"      ❌ NLPTransformerClassifier échec : {e}")
        nlp_ok = False

    # 5. Tester CNNBiLSTMTransformerClassifier
    print("\n  [5] Vérification avec CNNBiLSTMTransformerClassifier...")
    hybrid_model = CNNBiLSTMTransformerClassifier(
        input_size=input_size,
        num_classes=num_classes,
        seq_length=seq_length,
        config=CNN_BILSTM_TRANSFORMER_CONFIG
    )
    try:
        out = hybrid_model(x_tensor)
        print(f"      ✅ CNNBiLSTMTransformerClassifier : input {x_tensor.shape} → output {out.shape}")
        hybrid_ok = True
    except Exception as e:
        print(f"      ❌ CNNBiLSTMTransformerClassifier échec : {e}")
        hybrid_ok = False

    # Résumé
    print("\n" + "=" * 70)
    print("  RÉSUMÉ DE LA VÉRIFICATION")
    print("=" * 70)
    print(f"    Tokenizer créé           : {'✅ OK' if tokenizer else '❌ Échec'}")
    print(f"    Vocabulaire size         : {vocab_size:,} tokens")
    print(f"    TransformerClassifier    : {'✅ Compatible (features brutes)' if transformer_ok else '❌ Incompatible'}")
    print(f"    NLPTransformerClassifier : {'✅ Compatible (tokens)' if nlp_ok else '❌ Incompatible'}")
    print(f"    CNNBiLSTMTransformer     : {'✅ Compatible (features brutes)' if hybrid_ok else '❌ Incompatible'}")
    print("=" * 70 + "\n")

    return {
        'tokenizer': tokenizer,
        'transformer_ok': transformer_ok,
        'nlp_transformer_ok': nlp_ok,
        'hybrid_ok': hybrid_ok,
        'vocab_size': vocab_size
    }

if DATASETS in ['json', 'both']:
    tokenization_results_json = verify_tokenization_for_transformers_json(seq_length=SEQ_LENGTH, input_size=36)
else:
    print('Skipping tokenization verification for JSON')


In [ ]:
# ─── MODEL 5: Transformer on JSON ────────────────────────────────
MODEL = 'transformer'
print(f'\n{"#"*80}')
print(f'  PHASE 3b — {MODEL.upper()} on JSON (MAC address labeling)')
print(f'{"#"*80}\n')

log_memory(f'before_{MODEL}_json')

if DATASETS in ['json', 'both']:
    train_model(MODEL, 'json', JSON_DATA_DIR)
    verify_drive_save(MODEL, 'json')
else:
    print('Skipping JSON dataset')

print(f'\n✅ {MODEL.upper()} on JSON DONE')


In [ ]:
# ─── MODEL 6: CNN-BiLSTM-Transformer on JSON ─────────────────────
MODEL = 'cnn_bilstm_transformer'
print(f'\n{"#"*80}')
print(f'  PHASE 3b — {MODEL.upper()} on JSON (MAC address labeling)')
print(f'{"#"*80}\n')

log_memory(f'before_{MODEL}_json')

if DATASETS in ['json', 'both']:
    train_model(MODEL, 'json', JSON_DATA_DIR)
    verify_drive_save(MODEL, 'json')
else:
    print('Skipping JSON dataset')

aggressive_cleanup()

print(f'\n✅ {MODEL.upper()} on JSON DONE')

print(f'\n{"="*80}')
print('  ✅ ALL 6 MODELS TRAINED ON BOTH DATASETS — COMPLETE!')
print(f'{"="*80}')


# 📊 Phase 4 : Visualisation de TOUS les Résultats

Affiche les plots générés pendant l'entraînement — directement depuis Drive.

Les fichiers suivants sont générés pour chaque modèle, chaque dataset et chaque phase :
- `fig_device_identification_scores_phaseN.png` — P/R/F1 par appareil
- `fig_adversarial_effect_phaseN.png` — Impact des attaques par appareil
- `fig_robustness_summary_phaseN.png` — Accuracy + Macro-F1 clean vs attaques
- `fig_training_history.png` — Courbes loss/accuracy

In [ ]:
# ─── Cell: Display ALL Plots ─────────────────────────────────────────────────
import json
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path
from IPython.display import display, HTML

results_dir = Path(DRIVE_RESULTS_DIR) / 'models'
if not results_dir.exists():
    print('No results found yet — run the training cells first.')
else:
    result_dirs = sorted(results_dir.iterdir())

    for rd in result_dirs:
        if not rd.is_dir():
            continue

        model_name = rd.name
        print(f"\n{'='*80}")
        print(f"  📁 {model_name}")
        print(f"{'='*80}")

        # Show results summary
        results_file = rd / 'results.json'
        if results_file.exists():
            with open(results_file) as f:
                res = json.load(f)
            print(f"  Model: {res.get('model_type', '?')} | Input: {res.get('input_size', '?')} | Classes: {res.get('num_classes', '?')}")
            print(f"  Clean Accuracy: {res.get('test_accuracy_clean', 0):.4f}")
            if 'clean_metrics' in res:
                cm = res['clean_metrics']
                print(f"  Macro F1: {cm.get('macro_f1', 0):.4f} | Macro P: {cm.get('macro_precision', 0):.4f} | Macro R: {cm.get('macro_recall', 0):.4f}")
            if 'robustness_ratios' in res:
                print(f"  Robustness Ratios: {res['robustness_ratios']}")

        # Display all plots
        plot_files = sorted(rd.glob('fig_*.png'))
        if plot_files:
            for pf in plot_files:
                print(f"\n  📊 {pf.name}")
                fig, ax = plt.subplots(figsize=(16, 8))
                img = mpimg.imread(str(pf))
                ax.imshow(img)
                ax.axis('off')
                ax.set_title(pf.stem.replace('_', ' ').title(), fontsize=14, fontweight='bold')
                plt.tight_layout()
                plt.show()
        else:
            print('  No plots found.')

        # Show curriculum report if available
        report_file = rd / 'curriculum_report.json'
        if report_file.exists():
            with open(report_file) as f:
                report = json.load(f)
            print(f"\n  📋 Curriculum Report:")
            for phase_key, phase_data in report.get('phases', {}).items():
                phase_num = phase_data.get('phase', '?')
                clean = phase_data.get('clean', {})
                feat = phase_data.get('feature_attack', {})
                pgd = phase_data.get('sequence_pgd', {})
                fgsm = phase_data.get('sequence_fgsm', {})
                print(f"    Phase {phase_num}:")
                print(f"      Clean  — Acc: {clean.get('accuracy', 0):.4f}  F1: {clean.get('f1_score', 0):.4f}  Macro-F1: {clean.get('macro_f1', 0):.4f}")
                if feat:
                    print(f"      FeatAdv — Acc: {feat.get('accuracy', 0):.4f}  F1: {feat.get('f1_score', 0):.4f}  RR: {feat.get('robustness_ratio', 0):.4f}")
                if pgd:
                    print(f"      SeqPGD — Acc: {pgd.get('accuracy', 0):.4f}  F1: {pgd.get('f1_score', 0):.4f}  RR: {pgd.get('robustness_ratio', 0):.4f}")
                if fgsm:
                    print(f"      SeqFGSM — Acc: {fgsm.get('accuracy', 0):.4f}  F1: {fgsm.get('f1_score', 0):.4f}  RR: {fgsm.get('robustness_ratio', 0):.4f}")

print('\n\n✅ All results displayed.')

# 📋 Phase 5 : Tableau Récapitulatif Multi-Modèles (CSV + JSON)

Compare les performances de tous les modèles entraînés sur les deux datasets.

In [ ]:
# ─── Cell: Comparative Summary Table ─────────────────────────────────────────
import json
from pathlib import Path

results_dir = Path(DRIVE_RESULTS_DIR) / 'models'
if not results_dir.exists():
    print('No results found.')
else:
    print(f"{'Model':<30} {'Clean Acc':>10} {'Macro F1':>10} {'Feat Adv':>10} {'Seq PGD':>10} {'Seq FGSM':>10} {'Hybrid':>10}")
    print(f"{'-'*100}")

    for rd in sorted(results_dir.iterdir()):
        rf = rd / 'results.json'
        if not rf.exists():
            continue
        with open(rf) as f:
            res = json.load(f)

        model = rd.name
        clean_acc = res.get('test_accuracy_clean', 0)
        macro_f1 = res.get('clean_metrics', {}).get('macro_f1', 0)

        adv = res.get('adversarial_results', {})
        feat_acc = adv.get('feature_level', {}).get('accuracy', 0)
        pgd_acc = adv.get('sequence_pgd', {}).get('accuracy', 0)
        fgsm_acc = adv.get('sequence_fgsm', {}).get('accuracy', 0)
        hybrid_acc = adv.get('hybrid', {}).get('accuracy', 0)

        print(f"{model:<30} {clean_acc:>10.4f} {macro_f1:>10.4f} {feat_acc:>10.4f} {pgd_acc:>10.4f} {fgsm_acc:>10.4f} {hybrid_acc:>10.4f}")

    print(f"{'-'*100}")
    print('\n✅ Comparison complete.')